### RAG Evaluation

In [57]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACKING"] = "true"


In [58]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [60]:
import truststore
truststore.inject_into_ssl()

In [61]:
# Default URLs
DEFAULT_DOC_URL = ["https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]


# Load documents from the URLs
docs = [WebBaseLoader(url).load() for url in DEFAULT_DOC_URL]
docs_list = [item for sublist in docs for item in sublist]

# load documents from the URLs
docs = [WebBaseLoader(url).load() for url in DEFAULT_DOC_URL]
docs_list = [item for sublist in docs for item in sublist]

# Intialize a text splitter with specified checnk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250,
    chunk_overlap=0,
)

# Split the documents into chunks
docs_splits = text_splitter.split_documents(docs_list)

# Add the document chunks to the "vector store" using OpenAIEmbeddings
vectorstore = InMemoryVectorStore.from_documents(
    documents = docs_splits, 
    embedding=OpenAIEmbeddings(),
)

# With Langchain we can easily turn any vector store into a retrieval component:
retriever = vectorstore.as_retriever(K=6)


In [62]:
retriever.invoke("what is agents")

[Document(id='6c2cd238-7a89-47d8-8673-85e8cc149e6c', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [63]:
import os
from langchain.chat_models import init_chat_model
llm = init_chat_model("openai:gpt-4o-mini")
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000018B645CFF20>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000018B5E3C9A90>, root_client=<openai.OpenAI object at 0x0000018B64A54510>

In [66]:
from langsmith import traceable

## Add decorator
@traceable()
def rag_bot(question: str) -> dict:
    # Relevant context
    docs = retriever.invoke(question)
    docs_string = " ".join([doc.page_content for doc in docs])

    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions. Keep the answer concise and relevant to the provided documents."

Documents:
{docs_string}
"""

    ai_msg = llm.invoke([
        ("system", instructions),
        ("human", question),
    ])

    return {"answer": ai_msg.content, "documents": docs}


In [67]:
rag_bot("what is agents")

{'answer': 'Agents, in the context of the provided documents, refer to autonomous entities powered by large language models (LLMs) that can perform tasks, make decisions, and interact in various environments. They utilize components such as planning, memory, and self-reflection to behave in a way that mimics human behavior and can respond to different situations based on past experiences. Generative agents, for example, are designed to create believable simulations of human interaction within a controlled environment.',
 'documents': [Document(id='6c2cd238-7a89-47d8-8673-85e8cc149e6c', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copie

### DataSet

In [69]:
from langsmith import Client

client = Client()

# Define the examples for the dataset
examples = [
    {
        "inputs": {"question": "How does the ReACt agent use self-reflection?"},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools"}
    },
    {
        "inputs": {"question": "what are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority bias, (2) Selection bias, and (3) Confirmation bias."}
    },
    {
        "inputs": {"question": "what are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks include (1) Evasion attacks, (2) Poisoning attacks, (3) Model inversion attacks, (4) Membership inference attacks, and (5) Backdoor attacks."}
    },
]

### create the dataset and example in Langsmith
dataset_name = "evaluation_rag_dataset"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
        dataset_id=dataset.id, 
        examples=examples,
    )


{'example_ids': ['3000a76b-5245-4607-b74d-511cfdb99abd',
  'd83af419-7c5f-4d71-8efa-3fe83d942e37',
  'f9dc8fbc-b2b9-46fe-9db5-1151cd2e5108'],
 'count': 3,
 'as_of': '2026-10-24T05:45:38.67627399Z'}

### Evaluators or metrics

1. Correctness: Response vs reference answer

Goal: Measure :how similar/correct is the RAG chain answer, releative to ground-truth answer"
Mode: Requires a ground truth (reference) answer supplied through a dataset
Evaluator : Use LLM-as-judge to assess answer correctness.

In [83]:
from typing_extensions import Annotated, TypedDict

## Correctness Output Schema

class CorrectnessGrade(TypedDict):
    explanation:Annotated[str,"Explain your reasoning for the score"]
    correct:Annotated[bool, "True if the answer is correct, false Otherwise"]

## correctness prompt

correctness_instructions = """
You are teacher grading a quiz.

You will be given a QUESTIONS, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER.

Here is the grade criteria to follow:

(1) Grade the student answer based only on their factual accuracy relative to the ground thruth answer.
(2) Ensure that the student answer doesn't contain any conflicting statements.
(3) It's is OK if the student answer contains more information than the ground thruth, as long as it is correct answer.
"""

from langchain_openai import ChatOpenAI
grader_llm=ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(CorrectnessGrade,
                                                                        method="json_schema", strict=True)
# Evaluator metrics
def correctness(inputs: dict, outputs: dict, reference_outputs:dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answer = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}
"""

# Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answer},
    ])

    return grade['correct']


### Relevance : Response vs input
The flow is similar to above , but we simply look at the inputs and outputs without needing the reference_outputs. Without a reference answer we can't grade accuracy, but can still grade relevance-as in , did the model address the user's question or not.

In [82]:
from typing_extensions import Annotated, TypedDict
from langchain_openai import ChatOpenAI

# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, "Explain your reasoning for the score"]
    relevant: Annotated[bool, "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions = """
You are a teacher grading a quiz.

You will be given a QUESTION and the STUDENT ANSWER.

Here is the grade criteria to follow:

(1) Grade the student answer based only on its relevance to the question.
(2) Ensure that the student answer addresses the question directly.
(3) It's OK if the student answer contains additional information, as long as it is relevant.

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain the reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset.
"""

# Grader LLM
relevance_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(
    RelevanceGrade,
    method="json_schema",
    strict=True,
)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"Question: {inputs['question']}\nStudent Answer: {outputs['answer']}"
    grade = relevance_llm.invoke([
        ("system", relevance_instructions),
        ("human", answer),
    ])
    return grade["relevant"]


### Groundedness : Response vs retrieved docs

Another useful way to evaluate responses without reference answer is to check if the response is justified by (or "grounded in") the retrieved documents

In [84]:
# Grade output schema

class GroundedGrade(TypedDict):
    explanation: Annotated[str, "Explain your reasoning for the score"]
    grounded: Annotated[bool, "Provide the score on whether the answer is grounded in the provided context"]

# Grade prompt
grounded_instructions = """
You are a teacher grading a quiz.

You will be given a QUESTION, FACT and the STUDENT ANSWER.

Here is the grade criteria to follow:

(1) Grade the student is grounded in the provided fact.
(2) Ensure that the student answer doesn't contain "hallucinated" information outside the scope of the FACT


Grounded:
(1) A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain the reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset.
"""

grounded_llm = ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(GroundedGrade, method="json_schema", strict=True)

#Evaludator
def groundedness(inputs: dict, outputs:dict) -> bool:
    """ A simple evaluator for RAG answer groundedness. """
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer =f"FACTS: {doc_string}\n STUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([
        {"role":"system", "content" : grounded_instructions},
        {"role":"user", "content": answer}])
    return grade["grounded"]

### Retrieval Relevance : Retrieved documents vs input

In [85]:
from typing_extensions import Annotated, TypedDict
from langchain_openai import ChatOpenAI

# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, "Explain your reasoning for the score"]
    relevant: Annotated[bool, "Whether the retrieved documents are relevant to the query"]

# Grade prompt
retrieval_relevance_instruction = """
You are evaluating whether retrieved facts are relevant to a question.

Return relevant=True if the facts contain keywords or semantic meaning related to the question, even if some details are unrelated. Otherwise, return relevant=False.
"""

# Grader LLM
retrieval_relevance_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(
    RetrievalRelevanceGrade,
    method="json_schema",
    strict=True,
)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """Evaluate whether the retrieved documents are relevant to the question."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"
    grade = retrieval_relevance_llm.invoke([
        ("system", retrieval_relevance_instruction),
        ("human", answer),
    ])
    return grade["relevant"]


### Run the evaluation 

In [86]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, gpt-4-0125-preview"}
)

# Explore results locallly as a dataframe if you have pandas installed
experiment_results.to_pandas()


View the evaluation results for experiment: 'rag-doc-relevance-4baecc92' at:
https://smith.langchain.com/o/ad944d8b-e2b1-4fa4-a22d-8054b1844931/datasets/9006bd6a-9060-450a-80c0-0386dcf1593b/compare?selectedSessions=71798e73-52c5-4671-b452-96699c11886c




3it [00:27,  9.01s/it]


,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,How does the ReACt agent use self-reflection?,The ReAct agent utilizes self-reflection by in...,[page_content='Self-reflection is a vital aspe...,None,"ReAct integrates reasoning and acting, perform...",True,False,True,True,2.896378,3000a76b-5245-4607-b74d-511cfdb99abd,01a16c75-5743-7441-9515-7834a0e8281c
1,what are the types of biases that can arise wi...,The types of biases that can arise with few-sh...,[page_content='Zero-shot and few-shot learning...,None,The biases that can arise with few-shot prompt...,False,True,True,True,2.515042,d83af419-7c5f-4d71-8efa-3fe83d942e37,01a16c75-7df4-7221-8815-624ecdf3d9b7
2,what are five types of adversarial attacks?,The five types of adversarial attacks mentione...,[page_content='Black-box attacks assume that a...,None,Five types of adversarial attacks include (1) ...,False,True,True,True,1.796354,f9dc8fbc-b2b9-46fe-9db5-1151cd2e5108,01a16c75-a1db-79f3-8739-b51149835013
